<a href="https://colab.research.google.com/github/aounraza379/AutiSense-AI/blob/main/AutiSense_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq gradio openai-whisper gTTS
!apt install ffmpeg -y -q

In [ ]:
import os, time, whisper
import gradio as gr
from gtts import gTTS
from groq import Groq

# --- SETUP ---
GROQ_API_KEY = "use_api_here"
client = Groq(api_key=GROQ_API_KEY)
stt_model = whisper.load_model("tiny")

# --- SCENARIO PROMPT ENGINE ---
def get_scenario_prompt(stage, scenario):
    base_rules = """
    - Always stay in character
    - Use simple English
    - Keep replies 1-2 short sentences
    - Guide the child gently
    - If the user message is unclear, make a best guess and continue conversation.
    """

    stage_rules = f"""
    Current Stage: {stage}
    - GREETING: Welcome warmly
    - REQUEST: Respond to what they want/need
    - CLARIFICATION: Ask ONE simple follow-up question
    - CLOSING: End politely
    """

    if scenario == "teacher":
        role_desc = "You are Ms. Anna, a kind school teacher helping a student in a classroom. Encourage them to speak and ask simple learning questions."
    elif scenario == "restaurant":
        role_desc = "You are a waiter in a busy restaurant. Help the customer order food and ask about their menu choices."
    else: # Default: shop
        role_desc = "You are Emily, a friendly shopkeeper in a small store."

    return f"{role_desc}\n{base_rules}\n{stage_rules}"

# --- SESSION ---
class Session:
    def __init__(self):
        self.scenario = "shop" # Step 1: Default Scenario
        self.reset()

    def reset(self):
        self.history = []
        self.stage = "GREETING"
        return [], "Session reset! Say 'Hello' to start 👋", None, self.stage

    def update_stage(self, text):
        t = text.lower()
        if any(w in t for w in ["hello", "hi", "hey"]):
            self.stage = "REQUEST"
        elif any(w in t for w in ["want", "buy", "need", "order", "learn", "have"]):
            self.stage = "CLARIFICATION"
        elif any(w in t for w in ["bye", "thanks", "goodbye"]):
            self.stage = "CLOSING"

session = Session()

# --- TTS ---
def generate_speech(text):
    try:
        filename = f"voice_{int(time.time())}.mp3"
        gTTS(text=text, lang='en').save(filename)
        return filename
    except:
        return None

# --- UI ACTIONS ---
def change_scenario(choice):
    session.scenario = choice
    session.reset() # Soft reset to GREETING when scenario changes
    return f"Switched to {choice} mode! Let's start with a greeting."

# --- MAIN FLOW ---
def handle(audio, history):
    if not audio:
        return history, "Speak something 😊", None, session.stage

    text = stt_model.transcribe(audio)["text"].strip()

    if len(text) < 3:
        return history, "I didn't hear clearly 😊", None, session.stage

    session.update_stage(text)

    # FEEDBACK
    feedback = "Good! 👍" if "please" in text.lower() else "Try using 'please' 🙏"

    # Step 3: USE SCENARIO PROMPT
    system_prompt = get_scenario_prompt(session.stage, session.scenario)

    messages = [{"role": "system", "content": system_prompt}]
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": text})

    try:
        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages
        )
        ai = res.choices[0].message.content
    except:
        ai = "That sounds lovely! Would you like to tell me more about that?"

    audio_file = generate_speech(ai)

    history.append({"role": "user", "content": text})
    history.append({"role": "assistant", "content": ai})

    return history, feedback, audio_file, session.stage

# --- UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 AutiSense AI: Multi-Scenario Trainer")

    with gr.Row():
        with gr.Column(scale=2):
            chat = gr.Chatbot(type="messages", height=400)
            mic = gr.Audio(sources=["microphone"], type="filepath")
            audio_playback = gr.Audio(autoplay=True)

        with gr.Column(scale=1):
            # Step 4: ADD UI DROPDOWN
            scenario_select = gr.Dropdown(
                ["shop", "teacher", "restaurant"],
                value="shop",
                label="Choose Scenario"
            )
            feedback_label = gr.Label(value="Select a scenario and say Hello!")
            stage_box = gr.Textbox(label="Conversation Stage", value="GREETING", interactive=False)
            reset_btn = gr.Button("Reset Conversation")

    # Step 5: BINDINGS
    scenario_select.change(change_scenario, inputs=scenario_select, outputs=feedback_label)

    mic.change(handle, [mic, chat], [chat, feedback_label, audio_playback, stage_box])

    reset_btn.click(session.reset, None, [chat, feedback_label, audio_playback, stage_box])

demo.launch()